In [ ]:
import torch
import pytorch_lightning as pl
import torch
import matplotlib.pyplot as plt
import numpy as np





1. "trilinear" upsampling method
2. Three Output Channels instead of One to model background, liver and tumor



In [9]:
class DoubleConv(torch.nn.Module):
 
    def __init__(self, in_channels, out_channels):
        
        super().__init__()
        self.step = torch.nn.Sequential(torch.nn.Conv3d(in_channels, out_channels, 3, padding=1),
                                        torch.nn.ReLU(),
                                        torch.nn.Conv3d(out_channels, out_channels, 3, padding=1),
                                        torch.nn.ReLU())
        
    def forward(self, X):
        return self.step(X)


In [10]:

class UNet(torch.nn.Module):
   

    def __init__(self):
       
        super().__init__()
        
        
        ############# DOWN #####################
        self.layer1 = DoubleConv(1, 32)
        self.layer2 = DoubleConv(32, 64)
        self.layer3 = DoubleConv(64, 128)
        self.layer4 = DoubleConv(128, 256)

        #########################################

        ############## UP #######################
        self.layer5 = DoubleConv(256 + 128, 128)
        self.layer6 = DoubleConv(128+64, 64)
        self.layer7 = DoubleConv(64+32, 32)
        self.layer8 = torch.nn.Conv3d(32, 3, 1)  # Output: 3 values -> background, liver, tumor
        #########################################

        self.maxpool = torch.nn.MaxPool3d(2)

    def forward(self, x):
        
        ####### DownConv 1#########
        x1 = self.layer1(x)
        x1m = self.maxpool(x1)
        ###########################
        
        ####### DownConv 2#########        
        x2 = self.layer2(x1m)
        x2m = self.maxpool(x2)
        ###########################

        ####### DownConv 3#########        
        x3 = self.layer3(x2m)
        x3m = self.maxpool(x3)
        ###########################
        
        ##### Intermediate Layer ## 
        x4 = self.layer4(x3m)
        ###########################

        ####### UpCONV 1#########        
        x5 = torch.nn.Upsample(scale_factor=2, mode="trilinear")(x4)  # Upsample with a factor of 2
        x5 = torch.cat([x5, x3], dim=1)  # Skip-Connection
        x5 = self.layer5(x5)
        ###########################

        ####### UpCONV 2#########        
        x6 = torch.nn.Upsample(scale_factor=2, mode="trilinear")(x5)        
        x6 = torch.cat([x6, x2], dim=1)  # Skip-Connection    
        x6 = self.layer6(x6)
        ###########################
        
        ####### UpCONV 3#########        
        x7 = torch.nn.Upsample(scale_factor=2, mode="trilinear")(x6)
        x7 = torch.cat([x7, x1], dim=1)       
        x7 = self.layer7(x7)
        ###########################
        
        ####### Predicted segmentation#########        
        ret = self.layer8(x7)
        return ret

## Testing

In [11]:
model = UNet()

In [12]:
random_input = torch.randn(1, 1, 128, 128, 128)


In [13]:
with torch.no_grad():
    output = model(random_input)
assert output.shape == torch.Size([1, 3, 128, 128, 128])

In [15]:
import pytorch_lightning as pl
import torch
import matplotlib.pyplot as plt
import numpy as np

from model import UNet

class Segmenter(pl.LightningModule):
    def __init__(self):
        super().__init__()
        self.model = UNet()
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=1e-4)
        self.loss_fn = torch.nn.CrossEntropyLoss()

    def forward(self, x):
        return self.model(x)

    def configure_optimizers(self):
        return [self.optimizer]

In [16]:
from sklearn.metrics import classification_report
import numpy as np
import torch
import torchio as tio

# -------- Load trained model --------
model = Segmenter.load_from_checkpoint(
    "weights/epoch=97-step=25773.ckpt",
    weights_only=False
)
model = model.eval()

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)

# -------- Load one validation sample --------
IDX = 4
mask = val_dataset[IDX]["Label"]["data"]
imgs = val_dataset[IDX]["CT"]["data"]

# -------- Patch-based inference --------
grid_sampler = tio.inference.GridSampler(val_dataset[IDX], 96, (8, 8, 8))
aggregator = tio.inference.GridAggregator(grid_sampler)

patch_loader = torch.utils.data.DataLoader(grid_sampler, batch_size=4)

with torch.no_grad():
    for patches_batch in patch_loader:
        input_tensor = patches_batch['CT']["data"].to(device)
        locations = patches_batch[tio.LOCATION]

        outputs = model(input_tensor)
        aggregator.add_batch(outputs, locations)

output_tensor = aggregator.get_output_tensor()
pred = output_tensor.argmax(0)

# -------- Metrics --------
y_true = np.array(mask).squeeze().flatten()
y_pred = pred.cpu().numpy().squeeze().flatten()

y_true = y_true.astype(int)
y_pred = y_pred.astype(int)

report = classification_report(
    y_true,
    y_pred,
    labels=[0,1,2],
    target_names=["Background", "Liver", "Tumor"],
    digits=4
)

print(report)

c:\Users\Dell\AppData\Local\Programs\Python\Python311\Lib\site-packages\pytorch_lightning\utilities\migration\migration.py:208: You have multiple `ModelCheckpoint` callback states in this checkpoint, but we found state keys that would end up colliding with each other after an upgrade, which means we can't differentiate which of your checkpoint callbacks needs which states. At least one of your `ModelCheckpoint` callbacks will not be able to reload the state.
Lightning automatically upgraded your loaded checkpoint from v1.3.5 to v2.6.1. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint c:\healthcare\08-3D-Liver-Tumor-Segmentation - Copy\weights\epoch=97-step=25773.ckpt`


NameError: name 'val_dataset' is not defined